# T1-07 Data classification

**Time:** 09:20–10:30 &nbsp;|&nbsp; **Evidence:** classification score on CI-01…CI-13

**Notebook learning outcomes**

- Say what an **index** is, in CivicDesk words.
- Label a piece of text **INDEX** or **NEVER**.
- Explain a **canary**, and why clearance is not the same as “allowed in the index.”


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — the new words

Yesterday the model only saw what you typed. Today you will give it a **corpus**: a pile of approved CivicDesk paragraphs. Before you pile them, you classify.

| Word | Plain meaning | CivicDesk example |
|---|---|---|
| **Corpus** | The set of documents you are willing to search | Twelve fictional handbook chunks, not an EDMS export |
| **Chunk** | One retrieval unit (a paragraph or short section), not a 200-page PDF | `HB-VAC-C2` is one chunk |
| **Index** | The searchable structure over chunks (keyword overlap here; embeddings in production) | Only chunks with `index_ok: true` |
| **Metadata** | Labels on a chunk that are not the paragraph itself | `clearance`, `id`, `citation` (Dublin Core: Identifier, Title, Rights — PREP-02b) |
| **NOT READY** | Not ready for the pile (OCR junk, image-only PDF, missing id, unchecked scan) | Send back for a clean copy. **This scoring cell treats not-ready as NEVER.** |
| **Canary** | Planted sensitive text. If a reply quotes it, the exercise **fails** | National ID `999-00-1111`, name Maya Reid |
| **PII** | Information that identifies a person | CVs, burgess forms, health records, live tickets |

**INDEX** means: this text may be stored in the classroom search pile. **NEVER** means: do not copy it in, even “just to try.” Prep pack: `prep/data_cleaning_and_metadata.md`.

This is **N5** (data ban) and the start of **N4** (access). T1-10 is the access gate. Today is “does it enter the pile at all?”

![May this enter the CivicDesk index?](images/t2_index_or_not.jpg)


## Ministry link

| Your EOI type | What must never enter an index in *this* room |
|---|---|
| **Labour** | Real CVs, scores, names |
| **Health / THA** | Patient detail, LMS submissions, suicide-guideline *cases* (the *rule* N1 may be indexed) |
| **Finance** | Cabinet *bodies*. Class metadata is a later access question |
| **RDLG** | Burgess applications, household names, addresses |
| **TTDF** | Patrol notes, landing sites, tracks (AUX-05) |
| **Trade / Defence assets** | Live GLPI/serial dumps — query the register; do not index this month’s extract |
| **all** | The canary line. If you index it, you fail the drill |

**N14 woven:** someone must be able to say later “this chunk was / was not in the index.” That is why every chunk has an id.


## 0. Backend — demo, OLMo, or Phi

Retrieval always runs on this machine. This cell only chooses **what generates**.

| `BACKEND` | Use when |
|---|---|
| **`demo`** | No GPU, overnight, or the server is down. Canned / template replies. **Not** a live-model score. |
| **`olmo`** | Local OLMo is up (T1-01a / `docker compose`, port **8000**). Needs VRAM. |
| **`phi`** | Local Phi is up (port **8001**). Compose loads OLMo only unless you set `CIVICDESK_LOAD_MODELS=olmo,phi`. |

Change the one marked line and re-run **this** cell. Later generate cells in this kernel follow it. If the live call fails, the notebook continues with a labelled local fallback.


In [ ]:
# [CHANGE] where generate() goes. Retrieval still runs here either way.
BACKEND = "demo"  # "demo" | "olmo" | "phi"

import civicdesk_rag as rag

rag.set_backend(BACKEND)


## 1. Look at the handbook pile (including the canary)

`index_ok` is a flag on the chunk. `clearance` is who may *see* it later. A restricted chunk can still be in the index (T1-10). A prohibited canary must not be.


In [5]:
# [RUN]
import civicdesk_rag as rag

print("All chunks in data/handbooks.json\n")
rag.print_chunks(include_prohibited=True)
print()
print("Indexed only (index_ok True):", len(rag.indexed_chunks()), "of", len(rag.load_chunks()))


All chunks in data/handbooks.json

id           idx   src          ver  status      pii  train title
HB-VAC-C2    True  born-digital v3   current     False True  Vacancy Handbook v3 — criterion C2
HB-ICT-PWD   True  born-digital v2   current     False True  ICT knowledge article — passwords
HB-ATTACH-04 True  born-digital v4   current     False True  Service-request attachments overnight
HB-ATTACH-03 True  born-digital v3   superseded  False True  Service-request attachments overnight (withdrawn)
HB-NOTICE-N1 True  born-digital v1   current     False True  Public notice rule N1
HB-OPR-07    True  born-digital v1   current     False True  OPR procurement — no award
HB-EDMS-ACC  True  born-digital v2   current     False True  EDMS access classes
HB-ASSET-Q   True  born-digital v1   current     False True  Asset questions — register is source of truth
HB-MIN-02    True  born-digital v1   current     False True  Minutes extract rule
HB-CLASS-01  True  born-digital v1   current     False Tr

### Teaching note

`HB-CANARY` has `index_ok: false` and `clearance: prohibited`. The helper `retrieve()` skips it unless you pass `include_prohibited=True` (a drill, not a product feature).

`HB-CAB-X` is **restricted** but `index_ok: true`. That is deliberate. The index may know a classed minute exists. Displaying the body is T1-10. Finance-shaped: even that is generous — many rooms would keep Cabinet bodies **out** of the index entirely (see CI-05).


## From paper to a chunk

A scan is not a chunk. You need a checked text file, intake metadata, and a person who says the numbers are right. The PNG below was **rendered for class**. The OCR file next to it was **not** extracted live in this notebook.


In [8]:
# [RUN]
from pathlib import Path
import json

root = Path("data/scan_sample")
try:
    from IPython.display import Image, display
    display(Image(filename=str(root / "notice_scan.png")))
except Exception:
    print("(open data/scan_sample/notice_scan.png in the file browser)")
print("--- ocr_raw.txt ---")
print((root / "ocr_raw.txt").read_text(encoding="utf-8"))
print("--- ocr_meta.json ---")
print((root / "ocr_meta.json").read_text(encoding="utf-8"))
print("This PNG is a classroom render. The raw OCR text is a shipped file, not a live Tesseract run.")


<IPython.core.display.Image object>
--- ocr_raw.txt ---
CIVICDESK RECORDS UNIT — scan header
Notice to clerks — desk copies of paper forms

Paper forms received at the front desk must be scanned the same
day and the paper returned to the locked cabinet. Do not leave
comp-
leted forms in an unlocked tray overnight.

CivicDesk Notice SCN-01   page 1 of 1

--- ocr_meta.json ---
{
  "engine": "hand-made stand-in",
  "tesseract": false
}

This PNG is a classroom render. The raw OCR text is a shipped file, not a live Tesseract run.


In [9]:
# [CHANGE] turn rules on or off, then compare with the expected clean text.
RULES = {
    "drop_header_footer": True,
    "join_hyphenated": True,
    "join_wrapped_lines": True,
    "normalise_spaces": True,
}

from pathlib import Path
import civicdesk_rag as rag

raw = Path("data/scan_sample/ocr_raw.txt").read_text(encoding="utf-8")
expected = Path("data/scan_sample/ocr_clean.txt").read_text(encoding="utf-8").strip()
got, changed = rag.clean_ocr_text(raw, RULES)
print("rules that fired:", changed)
print("--- cleaned ---")
print(got)
print("--- expected ---")
print(expected)
print("character-level diff count:", sum(a != b for a, b in zip(got, expected)) + abs(len(got) - len(expected)))


rules that fired: ['drop_header_footer', 'join_hyphenated', 'join_wrapped_lines']
--- cleaned ---
Notice to clerks — desk copies of paper forms Paper forms received at the front desk must be scanned the same day and the paper returned to the locked cabinet. Do not leave completed forms in an unlocked tray overnight.
--- expected ---
Paper forms received at the front desk must be scanned the same day and the paper returned to the locked cabinet. Do not leave completed forms in an unlocked tray overnight.
character-level diff count: 206


### Scanning checklist

Prefer a **born-digital** file. If you must scan: 300 dpi or more (Tesseract's ImproveQuality guide), deskew, crop borders, keep the original image, record the engine and version, check names, dates and numbers by hand, and **never index an unchecked scan**.


In [11]:
# [OPTIONAL] live Tesseract only if the binary is on this machine.
import shutil
from pathlib import Path

if shutil.which("tesseract"):
    import pytesseract
    from PIL import Image
    print("tesseract:", shutil.which("tesseract"))
    print(pytesseract.image_to_string(Image.open("data/scan_sample/notice_scan.png")))
else:
    print("tesseract not installed. Using the shipped OCR files. Do not claim this notebook extracted the PNG live.")


tesseract not installed. Using the shipped OCR files. Do not claim this notebook extracted the PNG live.


In [12]:
# [CHANGE] intake record for CI-11, CI-12, CI-13. Copy field names from data/scan_sample/intake_example.json.
from pathlib import Path
import json
import civicdesk_rag as rag

example = json.loads(Path("data/scan_sample/intake_example.json").read_text(encoding="utf-8"))
print("example fields:", sorted(example))

INTAKE = {
    "CI-11": {**example, "id": "CI-11", "note": "unchecked scan — not ready to index"},
    "CI-12": {
        "id": "CI-12",
        "title": "HB-ATTACH-03 superseded overnight-laptop rule",
        "owner": "CivicDesk Records Unit",
        "source_type": "born-digital",
        "version": "v3",
        "status": "superseded",
        "index_ok": True,
        "ocr_checked_by": "not applicable",
        "note": "index it and filter status=current at query time",
    },
    "CI-13": {
        "id": "CI-13",
        "title": "fictional résumé",
        "source_type": "born-digital",
        "contains_personal_data": True,
        "index_ok": False,
        "train_ok": False,
        "note": "never",
    },
}
rag.save_log("T1-07_intake", INTAKE)


example fields: ['clearance', 'contains_personal_data', 'effective_date', 'id', 'index_ok', 'ocr_checked_by', 'ocr_engine', 'owner', 'retention', 'source_path', 'source_type', 'status', 'supersedes', 'title', 'train_ok', 'version']
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-07_intake.json


## 2. Classification sheet

Read CI-01…CI-13. Decide INDEX or NEVER **before** you open the next cell if you can.


In [14]:
# [RUN]
import civicdesk_rag as rag

rag.print_classify_sheet()


Label each item INDEX or NEVER. Then fill GUESSES in the next cell.

CI-01  [Labour / Health]
    CivicDesk Vacancy Handbook paragraph on diploma criterion C2.

CI-02  [Labour]
    A real applicant CV exported from a ministry e-recruitment folder.

CI-03  [Trade IMSD]
    ICT knowledge article: password reset is a human identity check.

CI-04  [TTDF]
    Last night's patrol notes with landing sites.

CI-05  [Finance]
    Cabinet Minute body on coastal security, full text.

CI-06  [Finance / MOWI / RDLG]
    CivicDesk Circular HB-ATTACH-04 on overnight attachments.

CI-07  [RDLG]
    Burgess application with household names and an address.

CI-08  [Health / THA]
    Public notice rule N1: no personal medical detail.

CI-09  [Defence assets / Trade]
    GLPI extract of live printer serials for this month.

CI-10  [all]
    A planted line: National ID 999-00-1111 Maya Reid.

CI-11  [records intake]
    A scanned CivicDesk front-desk notice. OCR has not been checked by a person yet.

CI-12

## 3. Score your labels

Allowed values: `index` or `never`. The cell ships the teaching key so Run All works. Cover it and try first if you want a real score.


In [16]:
# [CHANGE] replace with your labels to test yourself.
GUESSES = {
    "CI-01": "index",
    "CI-02": "never",
    "CI-03": "index",
    "CI-04": "never",
    "CI-05": "never",
    "CI-06": "index",
    "CI-07": "never",
    "CI-08": "index",
    "CI-09": "never",
    "CI-10": "never",
    "CI-11": "never",
    "CI-12": "index",
    "CI-13": "never",
}

import civicdesk_rag as rag

rag.score_classification(GUESSES)
rag.save_log("T1-07_classify", {"guesses": GUESSES})


CI-01  you=index   gold=index   OK  (Labour / Health)
CI-02  you=never   gold=never   OK  (Labour)
CI-03  you=index   gold=index   OK  (Trade IMSD)
CI-04  you=never   gold=never   OK  (TTDF)
CI-05  you=never   gold=never   OK  (Finance)
CI-06  you=index   gold=index   OK  (Finance / MOWI / RDLG)
CI-07  you=never   gold=never   OK  (RDLG)
CI-08  you=index   gold=index   OK  (Health / THA)
CI-09  you=never   gold=never   OK  (Defence assets / Trade)
CI-10  you=never   gold=never   OK  (all)
CI-11  you=never   gold=never   OK  (records intake)
CI-12  you=index   gold=index   OK  (Finance / MOWI / RDLG)
CI-13  you=never   gold=never   OK  (Labour)
score 13/13
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-07_classify.json


### After you run

A published **rule** can be indexed. A **person’s file** cannot. A **live extract** of a register cannot — you would be copying operational data into a search pile that the model might quote.

**Write-up:** name one thing from *your* overnight canvas that would be NEVER, in CivicDesk words (no real identifiers).


In [18]:
# [OPTIONAL] what retrieve() would do if someone left the canary searchable
import civicdesk_rag as rag

print("Normal retrieve (canary excluded):")
for h in rag.retrieve("Maya Reid national ID", k=5):
    print(" ", h["id"], "score", h["score"])
print()
print("Unsafe retrieve (include_prohibited=True) — classroom drill only:")
for h in rag.retrieve("Maya Reid national ID", k=5, include_prohibited=True):
    print(" ", h["id"], "score", h["score"], "index_ok", h.get("index_ok"))


Normal retrieve (canary excluded):

Unsafe retrieve (include_prohibited=True) — classroom drill only:
  HB-CANARY score 3 index_ok False
